In [ ]:
from __future__ import annotations

import base64
import json
import re
import time
from datetime import datetime, timezone
from io import BytesIO
from typing import Any, Literal
from uuid import uuid4

import google.auth
import pandas as pd
from google import genai
from google.api_core.exceptions import Conflict, NotFound, ResourceExhausted
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery
from google.cloud import storage
from google.genai import types
from IPython.display import Audio, Image as DisplayImage, Video, display
from PIL import Image as PILImage
from pydantic import BaseModel, ConfigDict, Field, field_validator

In [ ]:
PROJECT_ID = "leafy-guide-497515-m4"

GLOBAL_LOCATION = "global"
VIDEO_LOCATION = "us-central1"

BUCKET_NAME = "leafy-guide-497515-m4-vector-assets"

DATASET_ID = "exoplanet_wildlife_documentary_4k"

RUN_TABLE_ID = "documentary_runs"
SPECIES_TABLE_ID = "alien_species"
MEDIA_TABLE_ID = "documentary_media"
REVIEW_TABLE_ID = "documentary_reviews"

SPECIES_COUNT = 3

REASONING_MODEL_CANDIDATES = [
    "gemini-3.1-pro-preview",
    "gemini-2.5-pro",
    "gemini-2.5-flash",
]

IMAGE_GENERATION_ATTEMPTS = [
    {
        "model": "gemini-3-pro-image",
        "resolution": "4K",
    },
    {
        "model": "gemini-3.1-flash-image",
        "resolution": "4K",
    },
    {
        "model": "gemini-3-pro-image",
        "resolution": "2K",
    },
    {
        "model": "gemini-3.1-flash-image",
        "resolution": "2K",
    },
]

VIDEO_GENERATION_ATTEMPTS = [
    {
        "model": "veo-3.1-generate-001",
        "resolution": "4k",
    },
    {
        "model": "veo-3.1-generate-001",
        "resolution": "1080p",
    },
    {
        "model": "veo-3.1-fast-generate-001",
        "resolution": "1080p",
    },
]

LYRIA_MODEL_CANDIDATES = [
    "lyria-3-pro-preview",
    "lyria-3-clip-preview",
]

IMAGE_ASPECT_RATIO = "16:9"
VIDEO_ASPECT_RATIO = "16:9"
VIDEO_DURATION_SECONDS = 8

MAX_RETRIES = 5
BACKOFF_BASE_SECONDS = 8
BACKOFF_MAX_SECONDS = 90

IMAGE_DELAY_SECONDS = 8
AUDIO_DELAY_SECONDS = 8
VIDEO_DELAY_SECONDS = 15

GCS_IMAGE_PREFIX = "exoplanet-documentary/images"
GCS_AUDIO_PREFIX = "exoplanet-documentary/audio"
GCS_VIDEO_PREFIX = "exoplanet-documentary/videos"
GCS_MANIFEST_PREFIX = "exoplanet-documentary/manifests"
GCS_STAGING_PREFIX = "exoplanet-documentary/staging"
GCS_SUMMARY_PREFIX = "exoplanet-documentary/summaries"

print("Configuration loaded.")
print("Project:", PROJECT_ID)
print("Global location:", GLOBAL_LOCATION)
print("Video location:", VIDEO_LOCATION)
print("Bucket:", BUCKET_NAME)
print("Species count:", SPECIES_COUNT)
print("Preferred image resolution:", IMAGE_GENERATION_ATTEMPTS[0]["resolution"])
print("Preferred video resolution:", VIDEO_GENERATION_ATTEMPTS[0]["resolution"])

In [ ]:
class DocumentaryBrief(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        validate_assignment=True,
        str_strip_whitespace=True,
    )

    documentary_title: str = Field(
        min_length=5,
        max_length=160,
    )

    central_question: str = Field(
        min_length=30,
        max_length=800,
    )

    scientific_style: str = Field(
        min_length=30,
        max_length=1000,
    )

    cinematic_style: str = Field(
        min_length=30,
        max_length=1000,
    )

    target_audience: list[str] = Field(
        min_length=2,
        max_length=8,
    )

    required_habitats: list[
        Literal[
            "cloud_forest",
            "crystal_ocean",
            "twilight_desert",
        ]
    ]

    biological_constraints: list[str] = Field(
        min_length=4,
        max_length=15,
    )

    visual_constraints: list[str] = Field(
        min_length=4,
        max_length=15,
    )

    presentation_language: Literal[
        "English",
        "Polish",
    ]

    @field_validator("required_habitats")
    @classmethod
    def validate_required_habitats(
        cls,
        value: list[str],
    ) -> list[str]:
        if len(value) != SPECIES_COUNT:
            raise ValueError(
                f"Exactly {SPECIES_COUNT} habitats are required."
            )

        if len(value) != len(set(value)):
            raise ValueError(
                "Habitat values must be unique."
            )

        return value

In [ ]:
class AlienSpeciesConcept(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
    )

    species_id: str = Field(
        min_length=3,
        max_length=60,
    )

    common_name: str = Field(
        min_length=3,
        max_length=120,
    )

    scientific_name: str = Field(
        min_length=3,
        max_length=120,
    )

    habitat_type: Literal[
        "cloud_forest",
        "crystal_ocean",
        "twilight_desert",
    ]

    planet_name: str = Field(
        min_length=3,
        max_length=100,
    )

    planetary_conditions: str = Field(
        min_length=50,
        max_length=1200,
    )

    physical_anatomy: str = Field(
        min_length=80,
        max_length=1600,
    )

    locomotion: str = Field(
        min_length=40,
        max_length=900,
    )

    feeding_strategy: str = Field(
        min_length=40,
        max_length=900,
    )

    defensive_adaptations: str = Field(
        min_length=40,
        max_length=900,
    )

    social_behavior: str = Field(
        min_length=40,
        max_length=900,
    )

    ecological_role: str = Field(
        min_length=50,
        max_length=1200,
    )

    evolutionary_explanation: str = Field(
        min_length=80,
        max_length=1600,
    )

    documentary_hook: str = Field(
        min_length=20,
        max_length=300,
    )

    image_prompt: str = Field(
        min_length=150,
        max_length=3200,
    )

    video_prompt: str = Field(
        min_length=150,
        max_length=2800,
    )

    music_prompt: str = Field(
        min_length=100,
        max_length=1600,
    )

    visual_keywords: list[str] = Field(
        min_length=6,
        max_length=16,
    )


class AlienSpeciesPortfolio(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
    )

    documentary_title: str = Field(
        min_length=5,
        max_length=160,
    )

    scientific_premise: str = Field(
        min_length=100,
        max_length=1800,
    )

    narrative_arc: str = Field(
        min_length=100,
        max_length=1800,
    )

    species: list[AlienSpeciesConcept]

    @field_validator("species")
    @classmethod
    def validate_species(
        cls,
        value: list[AlienSpeciesConcept],
    ) -> list[AlienSpeciesConcept]:
        if len(value) != SPECIES_COUNT:
            raise ValueError(
                f"Expected exactly {SPECIES_COUNT} species."
            )

        species_ids = [
            species.species_id
            for species in value
        ]

        habitats = [
            species.habitat_type
            for species in value
        ]

        if len(species_ids) != len(set(species_ids)):
            raise ValueError(
                "species_id values must be unique."
            )

        if len(habitats) != len(set(habitats)):
            raise ValueError(
                "Each species must use a different habitat."
            )

        return value

In [ ]:
class MultimediaSpeciesReview(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
    )

    species_id: str

    image_summary: str
    video_summary: str
    soundtrack_summary: str

    biological_believability_score: int = Field(
        ge=1,
        le=10,
    )

    image_quality_score: int = Field(
        ge=1,
        le=10,
    )

    video_continuity_score: int = Field(
        ge=1,
        le=10,
    )

    soundtrack_fit_score: int = Field(
        ge=1,
        le=10,
    )

    documentary_value_score: int = Field(
        ge=1,
        le=10,
    )

    strongest_elements: list[str]
    visible_problems: list[str]
    recommended_edits: list[str]


class SpeciesFinalScore(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
    )

    species_id: str

    score: int = Field(
        ge=1,
        le=100,
    )

    reason: str


class DocumentaryEditDecision(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
    )

    final_title: str
    executive_summary: str
    recommended_sequence: list[str]
    opening_species_id: str
    closing_species_id: str
    strongest_species_id: str
    strongest_species_reason: str
    narrative_transitions: list[str]
    production_risks: list[str]
    next_iteration_actions: list[str]
    final_scores: list[SpeciesFinalScore]


print("Pydantic structured models created.")

In [ ]:
global_genai_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=GLOBAL_LOCATION,
)

video_genai_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=VIDEO_LOCATION,
)

storage_client = storage.Client(
    project=PROJECT_ID,
)

bucket = storage_client.bucket(
    BUCKET_NAME
)

bucket.reload()

bigquery_client = bigquery.Client(
    project=PROJECT_ID
)

credentials, authenticated_project = google.auth.default(
    scopes=[
        "https://www.googleapis.com/auth/cloud-platform",
    ]
)

authorized_session = AuthorizedSession(
    credentials
)

BUCKET_LOCATION = bucket.location

if BUCKET_LOCATION in {"US", "EU"}:
    BIGQUERY_LOCATION = BUCKET_LOCATION
else:
    BIGQUERY_LOCATION = BUCKET_LOCATION.lower()

print("Clients created.")
print("Authenticated project:", authenticated_project)
print("Bucket exists:", bucket.exists())
print("Bucket location:", BUCKET_LOCATION)
print("BigQuery location:", BIGQUERY_LOCATION)

In [ ]:
dataset_ref = bigquery.Dataset(
    f"{PROJECT_ID}.{DATASET_ID}"
)

dataset_ref.location = BIGQUERY_LOCATION

try:
    dataset = bigquery_client.create_dataset(
        dataset_ref
    )

    print(
        "Created dataset:",
        dataset.full_dataset_id,
    )

except Conflict:
    dataset = bigquery_client.get_dataset(
        dataset_ref
    )

    print(
        "Dataset already exists:",
        dataset.full_dataset_id,
    )

run_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{RUN_TABLE_ID}"
)

species_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{SPECIES_TABLE_ID}"
)

media_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{MEDIA_TABLE_ID}"
)

review_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{REVIEW_TABLE_ID}"
)

print("Run table:", run_table_ref)
print("Species table:", species_table_ref)
print("Media table:", media_table_ref)
print("Review table:", review_table_ref)

In [ ]:
run_schema = [
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "documentary_title",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "structured_brief_json",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "structured_portfolio_json",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "edit_decision_json",
        "STRING",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "model_selection_json",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]

species_schema = [
    bigquery.SchemaField(
        "species_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "common_name",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "scientific_name",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "planet_name",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "habitat_type",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "documentary_hook",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "species_json",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]

media_schema = [
    bigquery.SchemaField(
        "media_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "species_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "common_name",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "media_type",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "gcs_uri",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "mime_type",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "generation_model",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "requested_resolution",
        "STRING",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "width",
        "INTEGER",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "height",
        "INTEGER",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "duration_seconds",
        "INTEGER",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "size_bytes",
        "INTEGER",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "prompt",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]

review_schema = [
    bigquery.SchemaField(
        "review_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "species_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "review_type",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "review_json",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "review_model",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]

print("BigQuery schemas created.")

In [ ]:
def ensure_bigquery_table(
    table_ref: str,
    schema: list[bigquery.SchemaField],
) -> bigquery.Table:
    try:
        table = bigquery_client.get_table(
            table_ref
        )

        print(
            "Table already exists:",
            table.full_table_id,
        )

        return table

    except NotFound:
        table = bigquery.Table(
            table_ref,
            schema=schema,
        )

        created_table = bigquery_client.create_table(
            table
        )

        print(
            "Created table:",
            created_table.full_table_id,
        )

        return created_table


run_table = ensure_bigquery_table(
    run_table_ref,
    run_schema,
)

species_table = ensure_bigquery_table(
    species_table_ref,
    species_schema,
)

media_table = ensure_bigquery_table(
    media_table_ref,
    media_schema,
)

review_table = ensure_bigquery_table(
    review_table_ref,
    review_schema,
)

In [ ]:
def gcs_uri_from_blob_name(
    bucket_name: str,
    blob_name: str,
) -> str:
    return f"gs://{bucket_name}/{blob_name}"


def parse_gcs_uri(
    gcs_uri: str,
) -> tuple[str, str]:
    if not gcs_uri.startswith("gs://"):
        raise ValueError(
            f"Expected gs:// URI, got: {gcs_uri}"
        )

    without_scheme = gcs_uri.removeprefix(
        "gs://"
    )

    bucket_name, blob_name = without_scheme.split(
        "/",
        1,
    )

    return bucket_name, blob_name


def upload_text_to_gcs(
    text: str,
    *,
    blob_name: str,
    content_type: str,
) -> str:
    blob = bucket.blob(
        blob_name
    )

    blob.upload_from_string(
        text,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        BUCKET_NAME,
        blob_name,
    )


def upload_bytes_to_gcs(
    data: bytes,
    *,
    blob_name: str,
    content_type: str,
) -> str:
    blob = bucket.blob(
        blob_name
    )

    blob.upload_from_string(
        data,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        BUCKET_NAME,
        blob_name,
    )


def download_gcs_bytes(
    gcs_uri: str,
) -> bytes:
    source_bucket_name, blob_name = parse_gcs_uri(
        gcs_uri
    )

    source_bucket = storage_client.bucket(
        source_bucket_name
    )

    blob = source_bucket.blob(
        blob_name
    )

    return blob.download_as_bytes()


def get_gcs_blob_size(
    gcs_uri: str,
) -> int:
    source_bucket_name, blob_name = parse_gcs_uri(
        gcs_uri
    )

    source_bucket = storage_client.bucket(
        source_bucket_name
    )

    blob = source_bucket.blob(
        blob_name
    )

    blob.reload()

    return int(
        blob.size or 0
    )


def rows_to_ndjson(
    rows: list[dict[str, Any]],
) -> str:
    return "\n".join(
        json.dumps(
            row,
            ensure_ascii=False,
            default=str,
        )
        for row in rows
    )


def safe_slug(
    text: str,
) -> str:
    slug = re.sub(
        r"[^a-z0-9]+",
        "-",
        text.lower(),
    )

    slug = slug.strip("-")

    return slug[:80] or "asset"

In [ ]:
def display_gcs_image(
    gcs_uri: str,
    *,
    width: int = 720,
) -> None:
    image_bytes = download_gcs_bytes(
        gcs_uri
    )

    display(
        DisplayImage(
            data=image_bytes,
            width=width,
        )
    )


def display_gcs_audio(
    gcs_uri: str,
) -> None:
    audio_bytes = download_gcs_bytes(
        gcs_uri
    )

    display(
        Audio(
            data=audio_bytes,
            autoplay=False,
        )
    )


def display_gcs_video(
    gcs_uri: str,
    *,
    width: int = 720,
) -> None:
    video_bytes = download_gcs_bytes(
        gcs_uri
    )

    display(
        Video(
            data=video_bytes,
            embed=True,
            mimetype="video/mp4",
            width=width,
        )
    )

In [ ]:
def is_retryable_error(
    exc: Exception,
) -> bool:
    message = str(exc).lower()

    return (
        isinstance(exc, ResourceExhausted)
        or "resource exhausted" in message
        or "429" in message
        or "quota" in message
        or "rate limit" in message
        or "temporarily unavailable" in message
        or "503" in message
    )


def sleep_with_backoff(
    attempt: int,
) -> None:
    sleep_seconds = min(
        BACKOFF_MAX_SECONDS,
        BACKOFF_BASE_SECONDS * (2 ** (attempt - 1)),
    )

    print(
        f"Waiting {sleep_seconds} seconds before retry..."
    )

    time.sleep(
        sleep_seconds
    )


def call_with_backoff(
    function,
    *,
    operation_name: str,
    max_retries: int = MAX_RETRIES,
):
    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:
            return function()

        except Exception as exc:
            if (
                is_retryable_error(exc)
                and attempt < max_retries
            ):
                print(
                    operation_name,
                    "returned a temporary error.",
                )

                print(
                    "Attempt:",
                    attempt,
                    "/",
                    max_retries,
                )

                print(
                    "Error:",
                    str(exc)[:500],
                )

                sleep_with_backoff(
                    attempt
                )

                continue

            raise

    raise RuntimeError(
        f"{operation_name} failed after all retries."
    )

In [ ]:
def batch_load_rows_to_bigquery(
    *,
    rows: list[dict[str, Any]],
    table_ref: str,
    schema: list[bigquery.SchemaField],
    table_name: str,
    run_id: str,
) -> dict[str, Any]:
    if not rows:
        raise ValueError(
            f"No rows provided for table {table_name}."
        )

    ndjson_text = rows_to_ndjson(
        rows
    )

    staging_blob_name = (
        f"{GCS_STAGING_PREFIX}/"
        f"{run_id}/"
        f"{table_name}.ndjson"
    )

    staging_uri = upload_text_to_gcs(
        ndjson_text,
        blob_name=staging_blob_name,
        content_type="application/x-ndjson",
    )

    job_config = bigquery.LoadJobConfig(
        schema=schema,
        source_format=(
            bigquery.SourceFormat.NEWLINE_DELIMITED_JSON
        ),
        write_disposition=(
            bigquery.WriteDisposition.WRITE_APPEND
        ),
    )

    started_at = time.perf_counter()

    load_job = bigquery_client.load_table_from_uri(
        staging_uri,
        table_ref,
        location=dataset.location,
        job_config=job_config,
    )

    load_job.result()

    elapsed_seconds = (
        time.perf_counter()
        - started_at
    )

    result = {
        "table_name": table_name,
        "rows_loaded": len(rows),
        "staging_uri": staging_uri,
        "job_id": load_job.job_id,
        "elapsed_seconds": round(
            elapsed_seconds,
            3,
        ),
    }

    print("=" * 100)
    print(result)

    return result

In [ ]:
documentary_brief = DocumentaryBrief(
    documentary_title=(
        "Beyond Earth: Wildlife of Three Impossible Worlds"
    ),
    central_question=(
        "How might complex organisms evolve on planets with "
        "atmospheres, oceans, gravity and day cycles radically "
        "different from those found on Earth?"
    ),
    scientific_style=(
        "Speculative astrobiology grounded in evolutionary pressure, "
        "energy availability, atmospheric chemistry, biomechanics, "
        "ecological niches and convergent evolution."
    ),
    cinematic_style=(
        "Premium natural-history documentary, photorealistic macro "
        "cinematography, atmospheric environmental shots, realistic "
        "depth of field, carefully controlled motion and dramatic "
        "but scientifically believable lighting."
    ),
    target_audience=[
        "science documentary viewers",
        "science-fiction designers",
        "biology students",
        "technology enthusiasts",
    ],
    required_habitats=[
        "cloud_forest",
        "crystal_ocean",
        "twilight_desert",
    ],
    biological_constraints=[
        "Every organism must have a believable energy source",
        "Every anatomical feature must serve an ecological function",
        "Locomotion must match gravity and atmospheric density",
        "Feeding behavior must fit the local food chain",
        "The organism must not resemble a magical creature",
        "Avoid direct copies of existing Earth animals",
    ],
    visual_constraints=[
        "No readable text",
        "No logos",
        "No artificial watermark request",
        "No humanoid aliens",
        "No weapons",
        "No copyrighted creatures",
        "No identifiable human faces",
    ],
    presentation_language="English",
)

structured_brief_json = (
    documentary_brief.model_dump_json(
        indent=2
    )
)

print(structured_brief_json)

In [ ]:
ALIEN_SPECIES_PORTFOLIO_SCHEMA = {
    "type": "OBJECT",
    "required": [
        "documentary_title",
        "scientific_premise",
        "narrative_arc",
        "species",
    ],
    "properties": {
        "documentary_title": {
            "type": "STRING",
        },
        "scientific_premise": {
            "type": "STRING",
        },
        "narrative_arc": {
            "type": "STRING",
        },
        "species": {
            "type": "ARRAY",
            "minItems": SPECIES_COUNT,
            "maxItems": SPECIES_COUNT,
            "items": {
                "type": "OBJECT",
                "required": [
                    "species_id",
                    "common_name",
                    "scientific_name",
                    "habitat_type",
                    "planet_name",
                    "planetary_conditions",
                    "physical_anatomy",
                    "locomotion",
                    "feeding_strategy",
                    "defensive_adaptations",
                    "social_behavior",
                    "ecological_role",
                    "evolutionary_explanation",
                    "documentary_hook",
                    "image_prompt",
                    "video_prompt",
                    "music_prompt",
                    "visual_keywords",
                ],
                "properties": {
                    "species_id": {
                        "type": "STRING",
                    },
                    "common_name": {
                        "type": "STRING",
                    },
                    "scientific_name": {
                        "type": "STRING",
                    },
                    "habitat_type": {
                        "type": "STRING",
                        "enum": [
                            "cloud_forest",
                            "crystal_ocean",
                            "twilight_desert",
                        ],
                    },
                    "planet_name": {
                        "type": "STRING",
                    },
                    "planetary_conditions": {
                        "type": "STRING",
                    },
                    "physical_anatomy": {
                        "type": "STRING",
                    },
                    "locomotion": {
                        "type": "STRING",
                    },
                    "feeding_strategy": {
                        "type": "STRING",
                    },
                    "defensive_adaptations": {
                        "type": "STRING",
                    },
                    "social_behavior": {
                        "type": "STRING",
                    },
                    "ecological_role": {
                        "type": "STRING",
                    },
                    "evolutionary_explanation": {
                        "type": "STRING",
                    },
                    "documentary_hook": {
                        "type": "STRING",
                    },
                    "image_prompt": {
                        "type": "STRING",
                    },
                    "video_prompt": {
                        "type": "STRING",
                    },
                    "music_prompt": {
                        "type": "STRING",
                    },
                    "visual_keywords": {
                        "type": "ARRAY",
                        "items": {
                            "type": "STRING",
                        },
                    },
                },
            },
        },
    },
}

print("Species portfolio schema created.")

In [ ]:
def generate_structured_with_fallback(
    *,
    contents,
    response_schema: dict[str, Any],
    operation_name: str,
    temperature: float = 0.25,
) -> tuple[dict[str, Any], str]:
    errors = []

    for model_id in REASONING_MODEL_CANDIDATES:
        try:
            print(
                "Trying reasoning model:",
                model_id,
            )

            response = call_with_backoff(
                lambda model_id=model_id: (
                    global_genai_client.models.generate_content(
                        model=model_id,
                        contents=contents,
                        config=types.GenerateContentConfig(
                            response_mime_type="application/json",
                            response_schema=response_schema,
                            temperature=temperature,
                        ),
                    )
                ),
                operation_name=(
                    f"{operation_name} using {model_id}"
                ),
            )

            if not response.text:
                raise RuntimeError(
                    "Model returned an empty response."
                )

            parsed_json = json.loads(
                response.text
            )

            return (
                parsed_json,
                model_id,
            )

        except Exception as exc:
            errors.append(
                {
                    "model": model_id,
                    "error": str(exc)[:500],
                }
            )

            print(
                "Model failed:",
                model_id,
            )

            print(
                str(exc)[:500]
            )

    raise RuntimeError(
        f"{operation_name} failed for all models: "
        f"{json.dumps(errors, ensure_ascii=False)}"
    )

In [ ]:
def generate_alien_species_portfolio(
    brief: DocumentaryBrief,
) -> tuple[AlienSpeciesPortfolio, str]:
    prompt = f"""
You are an astrobiologist, evolutionary biologist,
wildlife documentary director and creature designer.

Create exactly {SPECIES_COUNT} fictional alien species.

Structured documentary brief:
{brief.model_dump_json(indent=2)}

Mandatory habitats:
1. cloud_forest
2. crystal_ocean
3. twilight_desert

Requirements:
- each species must come from a different planet
- each species must have a unique species_id
- anatomy must reflect gravity, atmosphere and habitat
- explain feeding, movement, defense and ecological role
- avoid direct copies of Earth animals
- make every species visually distinctive
- create a detailed 16:9 wildlife image prompt
- create a detailed image-to-video prompt
- create a Lyria soundtrack prompt
- soundtrack must be instrumental
- soundtrack must not include vocals or speech
- image and video must look like a premium nature documentary
- no text, logos, weapons or humanoid creatures
- output only valid JSON matching the schema
"""

    raw_portfolio, model_id = (
        generate_structured_with_fallback(
            contents=prompt,
            response_schema=(
                ALIEN_SPECIES_PORTFOLIO_SCHEMA
            ),
            operation_name=(
                "Generate alien species portfolio"
            ),
            temperature=0.5,
        )
    )

    portfolio = AlienSpeciesPortfolio.model_validate(
        raw_portfolio
    )

    expected_habitats = set(
        brief.required_habitats
    )

    actual_habitats = {
        species.habitat_type
        for species in portfolio.species
    }

    if actual_habitats != expected_habitats:
        raise ValueError(
            f"Expected habitats {expected_habitats}, "
            f"but received {actual_habitats}."
        )

    return portfolio, model_id


species_portfolio, portfolio_model_id = (
    generate_alien_species_portfolio(
        documentary_brief
    )
)

structured_portfolio_json = (
    species_portfolio.model_dump_json(
        indent=2
    )
)

print(
    "Portfolio model:",
    portfolio_model_id,
)

print(
    structured_portfolio_json
)

In [ ]:
run_id = str(
    uuid4()
)

run_timestamp = datetime.now(
    timezone.utc
)

species_rows = []

for species in species_portfolio.species:
    species_rows.append(
        {
            "species_id": species.species_id,
            "run_id": run_id,
            "common_name": species.common_name,
            "scientific_name": species.scientific_name,
            "planet_name": species.planet_name,
            "habitat_type": species.habitat_type,
            "documentary_hook": species.documentary_hook,
            "species_json": species.model_dump_json(
                indent=2
            ),
            "created_at": run_timestamp.isoformat(),
        }
    )

print("Run ID:", run_id)
print("Documentary:", species_portfolio.documentary_title)

display(
    pd.DataFrame(
        [
            {
                "species_id": row["species_id"],
                "common_name": row["common_name"],
                "scientific_name": row["scientific_name"],
                "planet": row["planet_name"],
                "habitat": row["habitat_type"],
            }
            for row in species_rows
        ]
    )
)

In [ ]:
def extract_image_bytes_from_response(
    response,
) -> tuple[bytes, str]:
    if not response.candidates:
        raise RuntimeError(
            "Image model returned no candidates."
        )

    content = response.candidates[0].content

    if content is None or not content.parts:
        raise RuntimeError(
            "Image model returned no content parts."
        )

    for part in content.parts:
        inline_data = getattr(
            part,
            "inline_data",
            None,
        )

        if inline_data is None:
            continue

        data = getattr(
            inline_data,
            "data",
            None,
        )

        if not data:
            continue

        if isinstance(data, str):
            image_bytes = base64.b64decode(
                data
            )
        else:
            image_bytes = bytes(
                data
            )

        mime_type = (
            getattr(
                inline_data,
                "mime_type",
                None,
            )
            or "image/png"
        )

        return (
            image_bytes,
            mime_type,
        )

    raise RuntimeError(
        "No generated image found in model response."
    )

In [ ]:
def convert_image_to_high_quality_jpeg(
    image_bytes: bytes,
) -> tuple[bytes, int, int]:
    source_buffer = BytesIO(
        image_bytes
    )

    with PILImage.open(
        source_buffer
    ) as image:
        rgb_image = image.convert(
            "RGB"
        )

        width, height = rgb_image.size

        output_buffer = BytesIO()

        rgb_image.save(
            output_buffer,
            format="JPEG",
            quality=96,
            subsampling=0,
            optimize=True,
        )

    return (
        output_buffer.getvalue(),
        width,
        height,
    )

In [ ]:
def generate_image_with_fallback(
    *,
    prompt: str,
) -> tuple[bytes, str, str, int, int]:
    errors = []

    for attempt_config in IMAGE_GENERATION_ATTEMPTS:
        model_id = attempt_config["model"]
        resolution = attempt_config["resolution"]

        try:
            print(
                "Trying image configuration:",
                model_id,
                resolution,
            )

            response = call_with_backoff(
                lambda model_id=model_id, resolution=resolution: (
                    global_genai_client.models.generate_content(
                        model=model_id,
                        contents=prompt,
                        config=types.GenerateContentConfig(
                            response_modalities=[
                                "TEXT",
                                "IMAGE",
                            ],
                            image_config=types.ImageConfig(
                                aspect_ratio=(
                                    IMAGE_ASPECT_RATIO
                                ),
                                image_size=resolution,
                            ),
                        ),
                    )
                ),
                operation_name=(
                    f"Generate {resolution} image "
                    f"with {model_id}"
                ),
            )

            raw_image_bytes, raw_mime_type = (
                extract_image_bytes_from_response(
                    response
                )
            )

            (
                jpeg_bytes,
                width,
                height,
            ) = convert_image_to_high_quality_jpeg(
                raw_image_bytes
            )

            print(
                "Generated dimensions:",
                width,
                "x",
                height,
            )

            return (
                jpeg_bytes,
                model_id,
                resolution,
                width,
                height,
            )

        except Exception as exc:
            errors.append(
                {
                    "model": model_id,
                    "resolution": resolution,
                    "error": str(exc)[:500],
                }
            )

            print(
                "Image attempt failed:",
                model_id,
                resolution,
            )

            print(
                str(exc)[:500]
            )

    raise RuntimeError(
        "All image-generation attempts failed: "
        + json.dumps(
            errors,
            ensure_ascii=False,
        )
    )

In [ ]:
def generate_species_image(
    species: AlienSpeciesConcept,
) -> dict[str, Any]:
    image_prompt = f"""
Create a photorealistic premium wildlife documentary frame.

Species:
{species.common_name}

Scientific name:
{species.scientific_name}

Planet:
{species.planet_name}

Habitat:
{species.habitat_type}

Planetary conditions:
{species.planetary_conditions}

Physical anatomy:
{species.physical_anatomy}

Locomotion:
{species.locomotion}

Feeding strategy:
{species.feeding_strategy}

Ecological role:
{species.ecological_role}

Specific visual direction:
{species.image_prompt}

Visual keywords:
{json.dumps(
    species.visual_keywords,
    ensure_ascii=False,
)}

Global scientific style:
{documentary_brief.scientific_style}

Global cinematic style:
{documentary_brief.cinematic_style}

Requirements:
- one clearly visible alien organism
- organism shown naturally inside its ecosystem
- physically believable anatomy
- realistic skin, scales, membranes or exoskeleton
- natural-history documentary cinematography
- high dynamic range
- cinematic foreground and background
- realistic depth of field
- no readable text
- no logos
- no human face
- no humanoid body
- no magical energy
- no weapon
- no existing copyrighted creature
"""

    (
        image_bytes,
        image_model_id,
        requested_resolution,
        width,
        height,
    ) = generate_image_with_fallback(
        prompt=image_prompt
    )

    blob_name = (
        f"{GCS_IMAGE_PREFIX}/"
        f"{run_id}/"
        f"{safe_slug(species.species_id)}.jpg"
    )

    image_gcs_uri = upload_bytes_to_gcs(
        image_bytes,
        blob_name=blob_name,
        content_type="image/jpeg",
    )

    return {
        "media_id": str(uuid4()),
        "run_id": run_id,
        "species_id": species.species_id,
        "common_name": species.common_name,
        "media_type": "species_image",
        "gcs_uri": image_gcs_uri,
        "mime_type": "image/jpeg",
        "generation_model": image_model_id,
        "requested_resolution": requested_resolution,
        "width": width,
        "height": height,
        "duration_seconds": None,
        "size_bytes": len(image_bytes),
        "prompt": image_prompt,
        "created_at": (
            datetime.now(timezone.utc).isoformat()
        ),
    }


image_rows = []

for index, species in enumerate(
    species_portfolio.species,
    start=1,
):
    print("=" * 100)

    print(
        f"Generating image {index}/{SPECIES_COUNT}:",
        species.common_name,
    )

    image_row = generate_species_image(
        species
    )

    image_rows.append(
        image_row
    )

    print(
        "Image saved to:",
        image_row["gcs_uri"],
    )

    if index < SPECIES_COUNT:
        time.sleep(
            IMAGE_DELAY_SECONDS
        )

assert len(image_rows) == 3

print(
    "Generated images:",
    len(image_rows),
)

In [ ]:
def generate_image_with_fallback(
    *,
    prompt: str,
) -> tuple[bytes, str, str, int, int]:
    errors = []

    for attempt_config in IMAGE_GENERATION_ATTEMPTS:
        model_id = attempt_config["model"]
        resolution = attempt_config["resolution"]

        try:
            print(
                "Trying image configuration:",
                model_id,
                resolution,
            )

            response = call_with_backoff(
                lambda model_id=model_id, resolution=resolution: (
                    global_genai_client.models.generate_content(
                        model=model_id,
                        contents=prompt,
                        config=types.GenerateContentConfig(
                            response_modalities=[
                                "TEXT",
                                "IMAGE",
                            ],
                            image_config=types.ImageConfig(
                                aspect_ratio=(
                                    IMAGE_ASPECT_RATIO
                                ),
                                image_size=resolution,
                            ),
                        ),
                    )
                ),
                operation_name=(
                    f"Generate {resolution} image "
                    f"with {model_id}"
                ),
            )

            raw_image_bytes, raw_mime_type = (
                extract_image_bytes_from_response(
                    response
                )
            )

            (
                jpeg_bytes,
                width,
                height,
            ) = convert_image_to_high_quality_jpeg(
                raw_image_bytes
            )

            print(
                "Generated dimensions:",
                width,
                "x",
                height,
            )

            return (
                jpeg_bytes,
                model_id,
                resolution,
                width,
                height,
            )

        except Exception as exc:
            errors.append(
                {
                    "model": model_id,
                    "resolution": resolution,
                    "error": str(exc)[:500],
                }
            )

            print(
                "Image attempt failed:",
                model_id,
                resolution,
            )

            print(
                str(exc)[:500]
            )

    raise RuntimeError(
        "All image-generation attempts failed: "
        + json.dumps(
            errors,
            ensure_ascii=False,
        )
    )

In [ ]:
for image_row in image_rows:
    print("=" * 100)

    print(
        "Species:",
        image_row["common_name"],
    )

    print(
        "Model:",
        image_row["generation_model"],
    )

    print(
        "Requested resolution:",
        image_row["requested_resolution"],
    )

    print(
        "Actual dimensions:",
        image_row["width"],
        "x",
        image_row["height"],
    )

    print(
        "Size:",
        round(
            image_row["size_bytes"] / 1_000_000,
            2,
        ),
        "MB",
    )

    print(
        "GCS URI:",
        image_row["gcs_uri"],
    )

    display_gcs_image(
        image_row["gcs_uri"],
        width=720,
    )

In [ ]:
def find_audio_output_recursively(
    value: Any,
) -> tuple[str, str] | None:
    if isinstance(value, dict):
        output_type = value.get(
            "type"
        )

        audio_data = value.get(
            "data"
        )

        mime_type = value.get(
            "mime_type"
        ) or value.get(
            "mimeType"
        )

        if (
            output_type == "audio"
            and isinstance(audio_data, str)
            and audio_data
        ):
            return (
                audio_data,
                mime_type or "audio/mpeg",
            )

        for nested_value in value.values():
            result = find_audio_output_recursively(
                nested_value
            )

            if result is not None:
                return result

    elif isinstance(value, list):
        for item in value:
            result = find_audio_output_recursively(
                item
            )

            if result is not None:
                return result

    return None

In [ ]:
LYRIA_INTERACTIONS_ENDPOINT = (
    "https://aiplatform.googleapis.com/"
    f"v1beta1/projects/{PROJECT_ID}/"
    "locations/global/interactions"
)


def generate_music_with_lyria(
    *,
    prompt: str,
    image_gcs_uri: str,
    image_mime_type: str,
) -> tuple[bytes, str, str]:
    errors = []

    for model_id in LYRIA_MODEL_CANDIDATES:
        payload = {
            "model": model_id,
            "input": [
                {
                    "type": "text",
                    "text": prompt,
                },
                {
                    "type": "image",
                    "mime_type": image_mime_type,
                    "uri": image_gcs_uri,
                },
            ],
        }

        try:
            print(
                "Trying Lyria model:",
                model_id,
            )

            response = call_with_backoff(
                lambda payload=payload: (
                    authorized_session.post(
                        LYRIA_INTERACTIONS_ENDPOINT,
                        json=payload,
                        timeout=600,
                    )
                ),
                operation_name=(
                    f"Generate music with {model_id}"
                ),
            )

            if not response.ok:
                raise RuntimeError(
                    f"Lyria HTTP {response.status_code}: "
                    f"{response.text[:1000]}"
                )

            response_json = response.json()

            audio_result = find_audio_output_recursively(
                response_json
            )

            if audio_result is None:
                raise RuntimeError(
                    "No audio output found in Lyria response: "
                    f"{json.dumps(response_json)[:1000]}"
                )

            audio_base64, mime_type = audio_result

            audio_bytes = base64.b64decode(
                audio_base64
            )

            return (
                audio_bytes,
                mime_type,
                model_id,
            )

        except Exception as exc:
            errors.append(
                {
                    "model": model_id,
                    "error": str(exc)[:500],
                }
            )

            print(
                "Lyria model failed:",
                model_id,
            )

            print(
                str(exc)[:500]
            )

    raise RuntimeError(
        "Music generation failed for all models: "
        + json.dumps(
            errors,
            ensure_ascii=False,
        )
    )

In [ ]:
def generate_species_soundtrack(
    species: AlienSpeciesConcept,
    image_row: dict[str, Any],
) -> dict[str, Any]:
    music_prompt = f"""
Create an original instrumental cinematic wildlife soundtrack.

Alien species:
{species.common_name}

Planet:
{species.planet_name}

Habitat:
{species.habitat_type}

Planetary environment:
{species.planetary_conditions}

Species behavior:
{species.social_behavior}

Documentary moment:
{species.documentary_hook}

Detailed musical direction:
{species.music_prompt}

Requirements:
- instrumental only
- no singing
- no spoken words
- no recognizable existing melody
- no imitation of a named artist
- rich environmental atmosphere
- premium natural-history documentary quality
- gradual musical development
- organic and synthetic textures
- support wonder rather than horror
- use the supplied species image as visual inspiration
"""

    (
        audio_bytes,
        audio_mime_type,
        audio_model_id,
    ) = generate_music_with_lyria(
        prompt=music_prompt,
        image_gcs_uri=image_row["gcs_uri"],
        image_mime_type=image_row["mime_type"],
    )

    extension = (
        "mp3"
        if "mpeg" in audio_mime_type
        else "wav"
    )

    blob_name = (
        f"{GCS_AUDIO_PREFIX}/"
        f"{run_id}/"
        f"{safe_slug(species.species_id)}."
        f"{extension}"
    )

    audio_gcs_uri = upload_bytes_to_gcs(
        audio_bytes,
        blob_name=blob_name,
        content_type=audio_mime_type,
    )

    return {
        "media_id": str(uuid4()),
        "run_id": run_id,
        "species_id": species.species_id,
        "common_name": species.common_name,
        "media_type": "species_soundtrack",
        "gcs_uri": audio_gcs_uri,
        "mime_type": audio_mime_type,
        "generation_model": audio_model_id,
        "requested_resolution": None,
        "width": None,
        "height": None,
        "duration_seconds": None,
        "size_bytes": len(audio_bytes),
        "prompt": music_prompt,
        "created_at": (
            datetime.now(timezone.utc).isoformat()
        ),
    }


audio_rows = []

for index, species in enumerate(
    species_portfolio.species,
    start=1,
):
    image_row = next(
        row
        for row in image_rows
        if row["species_id"] == species.species_id
    )

    print("=" * 100)

    print(
        f"Generating soundtrack {index}/{SPECIES_COUNT}:",
        species.common_name,
    )

    audio_row = generate_species_soundtrack(
        species,
        image_row,
    )

    audio_rows.append(
        audio_row
    )

    print(
        "Audio saved to:",
        audio_row["gcs_uri"],
    )

    if index < SPECIES_COUNT:
        time.sleep(
            AUDIO_DELAY_SECONDS
        )

assert len(audio_rows) == 3

print(
    "Generated soundtracks:",
    len(audio_rows),
)

In [ ]:
for audio_row in audio_rows:
    print("=" * 100)

    print(
        "Species:",
        audio_row["common_name"],
    )

    print(
        "Model:",
        audio_row["generation_model"],
    )

    print(
        "MIME type:",
        audio_row["mime_type"],
    )

    print(
        "Size:",
        round(
            audio_row["size_bytes"] / 1_000_000,
            2,
        ),
        "MB",
    )

    print(
        "GCS URI:",
        audio_row["gcs_uri"],
    )

    display_gcs_audio(
        audio_row["gcs_uri"]
    )

In [ ]:
def wait_for_video_operation(
    operation,
    *,
    poll_interval_seconds: int = 20,
    max_wait_seconds: int = 40 * 60,
):
    started_at = time.time()

    while not operation.done:
        elapsed_seconds = int(
            time.time() - started_at
        )

        if elapsed_seconds > max_wait_seconds:
            raise TimeoutError(
                "Video generation exceeded "
                "the maximum waiting time."
            )

        print(
            "Video generation running.",
            "Elapsed seconds:",
            elapsed_seconds,
        )

        time.sleep(
            poll_interval_seconds
        )

        operation = (
            video_genai_client.operations.get(
                operation
            )
        )

    return operation


def extract_video_uri(
    operation,
) -> str:
    operation_error = getattr(
        operation,
        "error",
        None,
    )

    if operation_error:
        raise RuntimeError(
            f"Video generation failed: "
            f"{operation_error}"
        )

    result = getattr(
        operation,
        "result",
        None,
    )

    if result is None:
        result = getattr(
            operation,
            "response",
            None,
        )

    if result is None:
        raise RuntimeError(
            "Video operation returned no result."
        )

    generated_videos = getattr(
        result,
        "generated_videos",
        None,
    )

    if not generated_videos:
        raise RuntimeError(
            "Video operation returned no generated videos."
        )

    video_object = generated_videos[0].video

    video_uri = (
        getattr(
            video_object,
            "uri",
            None,
        )
        or getattr(
            video_object,
            "gcs_uri",
            None,
        )
    )

    if not video_uri:
        raise RuntimeError(
            "Could not extract video URI."
        )

    return video_uri

In [ ]:
def generate_video_with_fallback(
    *,
    prompt: str,
    image_gcs_uri: str,
    image_mime_type: str,
    output_prefix: str,
) -> tuple[str, str, str]:
    errors = []

    for attempt_config in VIDEO_GENERATION_ATTEMPTS:
        model_id = attempt_config["model"]
        resolution = attempt_config["resolution"]

        try:
            print(
                "Trying video configuration:",
                model_id,
                resolution,
            )

            operation = call_with_backoff(
                lambda model_id=model_id, resolution=resolution: (
                    video_genai_client.models.generate_videos(
                        model=model_id,
                        prompt=prompt,
                        image=types.Image(
                            gcs_uri=image_gcs_uri,
                            mime_type=image_mime_type,
                        ),
                        config=types.GenerateVideosConfig(
                            number_of_videos=1,
                            duration_seconds=(
                                VIDEO_DURATION_SECONDS
                            ),
                            aspect_ratio=(
                                VIDEO_ASPECT_RATIO
                            ),
                            resolution=resolution,
                            output_gcs_uri=output_prefix,
                        ),
                    )
                ),
                operation_name=(
                    f"Start {resolution} video "
                    f"with {model_id}"
                ),
            )

            completed_operation = (
                wait_for_video_operation(
                    operation
                )
            )

            video_uri = extract_video_uri(
                completed_operation
            )

            return (
                video_uri,
                model_id,
                resolution,
            )

        except Exception as exc:
            errors.append(
                {
                    "model": model_id,
                    "resolution": resolution,
                    "error": str(exc)[:500],
                }
            )

            print(
                "Video attempt failed:",
                model_id,
                resolution,
            )

            print(
                str(exc)[:500]
            )

    raise RuntimeError(
        "All video-generation attempts failed: "
        + json.dumps(
            errors,
            ensure_ascii=False,
        )
    )

In [ ]:
def generate_species_video(
    species: AlienSpeciesConcept,
    image_row: dict[str, Any],
) -> dict[str, Any]:
    video_prompt = f"""
Create one continuous eight-second wildlife documentary shot.

Species:
{species.common_name}

Planet:
{species.planet_name}

Habitat:
{species.habitat_type}

Physical anatomy:
{species.physical_anatomy}

Natural locomotion:
{species.locomotion}

Feeding strategy:
{species.feeding_strategy}

Social behavior:
{species.social_behavior}

Specific motion direction:
{species.video_prompt}

Requirements:
- preserve the exact organism design from the source image
- preserve the same ecosystem and lighting
- show natural animal behavior
- use slow premium wildlife cinematography
- physically plausible body movement
- physically plausible environmental movement
- no sudden cuts
- no morphing anatomy
- no duplicated limbs
- no disappearing body parts
- no readable text
- no logos
- no humans
- no weapon
- no dialogue
- photorealistic natural-history documentary quality
"""

    output_prefix = (
        f"gs://{BUCKET_NAME}/"
        f"{GCS_VIDEO_PREFIX}/"
        f"{run_id}/"
        f"{safe_slug(species.species_id)}/"
    )

    (
        video_gcs_uri,
        video_model_id,
        requested_resolution,
    ) = generate_video_with_fallback(
        prompt=video_prompt,
        image_gcs_uri=image_row["gcs_uri"],
        image_mime_type=image_row["mime_type"],
        output_prefix=output_prefix,
    )

    video_size = get_gcs_blob_size(
        video_gcs_uri
    )

    return {
        "media_id": str(uuid4()),
        "run_id": run_id,
        "species_id": species.species_id,
        "common_name": species.common_name,
        "media_type": "species_video",
        "gcs_uri": video_gcs_uri,
        "mime_type": "video/mp4",
        "generation_model": video_model_id,
        "requested_resolution": requested_resolution,
        "width": None,
        "height": None,
        "duration_seconds": VIDEO_DURATION_SECONDS,
        "size_bytes": video_size,
        "prompt": video_prompt,
        "created_at": (
            datetime.now(timezone.utc).isoformat()
        ),
    }


video_rows = []

for index, species in enumerate(
    species_portfolio.species,
    start=1,
):
    image_row = next(
        row
        for row in image_rows
        if row["species_id"] == species.species_id
    )

    print("=" * 100)

    print(
        f"Generating video {index}/{SPECIES_COUNT}:",
        species.common_name,
    )

    video_row = generate_species_video(
        species,
        image_row,
    )

    video_rows.append(
        video_row
    )

    print(
        "Video saved to:",
        video_row["gcs_uri"],
    )

    if index < SPECIES_COUNT:
        time.sleep(
            VIDEO_DELAY_SECONDS
        )

assert len(video_rows) == 3

print(
    "Generated videos:",
    len(video_rows),
)

In [ ]:
for video_row in video_rows:
    print("=" * 100)

    print(
        "Species:",
        video_row["common_name"],
    )

    print(
        "Model:",
        video_row["generation_model"],
    )

    print(
        "Requested resolution:",
        video_row["requested_resolution"],
    )

    print(
        "Duration:",
        video_row["duration_seconds"],
        "seconds",
    )

    print(
        "Size:",
        round(
            video_row["size_bytes"] / 1_000_000,
            2,
        ),
        "MB",
    )

    print(
        "GCS URI:",
        video_row["gcs_uri"],
    )

    display_gcs_video(
        video_row["gcs_uri"],
        width=720,
    )

In [ ]:
MULTIMEDIA_REVIEW_SCHEMA = {
    "type": "OBJECT",
    "required": [
        "species_id",
        "image_summary",
        "video_summary",
        "soundtrack_summary",
        "biological_believability_score",
        "image_quality_score",
        "video_continuity_score",
        "soundtrack_fit_score",
        "documentary_value_score",
        "strongest_elements",
        "visible_problems",
        "recommended_edits",
    ],
    "properties": {
        "species_id": {
            "type": "STRING",
        },
        "image_summary": {
            "type": "STRING",
        },
        "video_summary": {
            "type": "STRING",
        },
        "soundtrack_summary": {
            "type": "STRING",
        },
        "biological_believability_score": {
            "type": "INTEGER",
            "minimum": 1,
            "maximum": 10,
        },
        "image_quality_score": {
            "type": "INTEGER",
            "minimum": 1,
            "maximum": 10,
        },
        "video_continuity_score": {
            "type": "INTEGER",
            "minimum": 1,
            "maximum": 10,
        },
        "soundtrack_fit_score": {
            "type": "INTEGER",
            "minimum": 1,
            "maximum": 10,
        },
        "documentary_value_score": {
            "type": "INTEGER",
            "minimum": 1,
            "maximum": 10,
        },
        "strongest_elements": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "visible_problems": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "recommended_edits": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
    },
}

print("Multimedia review schema created.")

In [ ]:
def find_species(
    species_id: str,
) -> AlienSpeciesConcept:
    for species in species_portfolio.species:
        if species.species_id == species_id:
            return species

    raise KeyError(
        f"Unknown species_id: {species_id}"
    )


def review_species_multimedia(
    species: AlienSpeciesConcept,
    image_row: dict[str, Any],
    audio_row: dict[str, Any],
    video_row: dict[str, Any],
) -> tuple[MultimediaSpeciesReview, str]:
    prompt = f"""
You are an astrobiologist, wildlife documentary editor,
visual-effects supervisor and music supervisor.

Evaluate all three generated assets for one alien species.

Structured species concept:
{species.model_dump_json(indent=2)}

Image metadata:
{json.dumps(
    image_row,
    indent=2,
    ensure_ascii=False,
    default=str,
)}

Video metadata:
{json.dumps(
    video_row,
    indent=2,
    ensure_ascii=False,
    default=str,
)}

Soundtrack metadata:
{json.dumps(
    audio_row,
    indent=2,
    ensure_ascii=False,
    default=str,
)}

Evaluate:
- biological believability
- anatomical consistency between image and video
- image quality
- video motion and temporal continuity
- soundtrack fit
- overall documentary value
- visible generation problems
- practical editing improvements

The returned species_id must be exactly:
{species.species_id}

Return only valid JSON matching the schema.
"""

    raw_review, review_model_id = (
        generate_structured_with_fallback(
            contents=[
                types.Part.from_uri(
                    file_uri=image_row["gcs_uri"],
                    mime_type=image_row["mime_type"],
                ),
                types.Part.from_uri(
                    file_uri=video_row["gcs_uri"],
                    mime_type=video_row["mime_type"],
                ),
                types.Part.from_uri(
                    file_uri=audio_row["gcs_uri"],
                    mime_type=audio_row["mime_type"],
                ),
                prompt,
            ],
            response_schema=(
                MULTIMEDIA_REVIEW_SCHEMA
            ),
            operation_name=(
                f"Review multimedia for "
                f"{species.common_name}"
            ),
            temperature=0.15,
        )
    )

    review = MultimediaSpeciesReview.model_validate(
        raw_review
    )

    if review.species_id != species.species_id:
        raise ValueError(
            "Review returned an incorrect species_id."
        )

    return (
        review,
        review_model_id,
    )


species_reviews: dict[
    str,
    MultimediaSpeciesReview,
] = {}

review_rows = []

for species in species_portfolio.species:
    image_row = next(
        row
        for row in image_rows
        if row["species_id"] == species.species_id
    )

    audio_row = next(
        row
        for row in audio_rows
        if row["species_id"] == species.species_id
    )

    video_row = next(
        row
        for row in video_rows
        if row["species_id"] == species.species_id
    )

    print("=" * 100)

    print(
        "Reviewing:",
        species.common_name,
    )

    review, review_model_id = (
        review_species_multimedia(
            species,
            image_row,
            audio_row,
            video_row,
        )
    )

    species_reviews[
        species.species_id
    ] = review

    review_rows.append(
        {
            "review_id": str(uuid4()),
            "run_id": run_id,
            "species_id": species.species_id,
            "review_type": "multimedia_species_review",
            "review_json": review.model_dump_json(
                indent=2
            ),
            "review_model": review_model_id,
            "created_at": (
                datetime.now(timezone.utc).isoformat()
            ),
        }
    )

    print(
        review.model_dump_json(
            indent=2
        )
    )

assert len(species_reviews) == 3

In [ ]:
DOCUMENTARY_EDIT_SCHEMA = {
    "type": "OBJECT",
    "required": [
        "final_title",
        "executive_summary",
        "recommended_sequence",
        "opening_species_id",
        "closing_species_id",
        "strongest_species_id",
        "strongest_species_reason",
        "narrative_transitions",
        "production_risks",
        "next_iteration_actions",
        "final_scores",
    ],
    "properties": {
        "final_title": {
            "type": "STRING",
        },
        "executive_summary": {
            "type": "STRING",
        },
        "recommended_sequence": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "opening_species_id": {
            "type": "STRING",
        },
        "closing_species_id": {
            "type": "STRING",
        },
        "strongest_species_id": {
            "type": "STRING",
        },
        "strongest_species_reason": {
            "type": "STRING",
        },
        "narrative_transitions": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "production_risks": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "next_iteration_actions": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "final_scores": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "required": [
                    "species_id",
                    "score",
                    "reason",
                ],
                "properties": {
                    "species_id": {
                        "type": "STRING",
                    },
                    "score": {
                        "type": "INTEGER",
                        "minimum": 1,
                        "maximum": 100,
                    },
                    "reason": {
                        "type": "STRING",
                    },
                },
            },
        },
    },
}

print("Documentary edit schema created.")

In [ ]:
def generate_documentary_edit_decision(
) -> tuple[DocumentaryEditDecision, str]:
    evidence = []

    for species in species_portfolio.species:
        evidence.append(
            {
                "species": species.model_dump(
                    mode="json"
                ),
                "review": species_reviews[
                    species.species_id
                ].model_dump(
                    mode="json"
                ),
                "media": [
                    {
                        "media_type": row["media_type"],
                        "gcs_uri": row["gcs_uri"],
                        "model": row["generation_model"],
                        "resolution": row[
                            "requested_resolution"
                        ],
                    }
                    for row in [
                        *image_rows,
                        *audio_rows,
                        *video_rows,
                    ]
                    if (
                        row["species_id"]
                        == species.species_id
                    )
                ],
            }
        )

    allowed_species_ids = [
        species.species_id
        for species in species_portfolio.species
    ]

    prompt = f"""
You are the lead editor of a premium
exoplanet wildlife documentary.

Documentary premise:
{species_portfolio.scientific_premise}

Narrative arc:
{species_portfolio.narrative_arc}

Structured production evidence:
{json.dumps(
    evidence,
    indent=2,
    ensure_ascii=False,
    default=str,
)}

Allowed species IDs:
{json.dumps(
    allowed_species_ids,
    indent=2,
)}

Create the final documentary edit decision.

Rules:
- recommended_sequence must contain every species ID exactly once
- opening_species_id must be an allowed ID
- closing_species_id must be an allowed ID
- strongest_species_id must be an allowed ID
- final_scores must contain every species ID exactly once
- each score must be between 1 and 100
- use the multimedia reviews as evidence
- create meaningful transitions between habitats
- identify technical generation problems
- propose practical next-generation improvements
- return only valid JSON matching the schema
"""

    raw_decision, decision_model_id = (
        generate_structured_with_fallback(
            contents=prompt,
            response_schema=DOCUMENTARY_EDIT_SCHEMA,
            operation_name=(
                "Generate final documentary edit decision"
            ),
            temperature=0.2,
        )
    )

    decision = DocumentaryEditDecision.model_validate(
        raw_decision
    )

    allowed_set = set(
        allowed_species_ids
    )

    if set(decision.recommended_sequence) != allowed_set:
        raise ValueError(
            "recommended_sequence must contain "
            "all species IDs exactly once."
        )

    for selected_id in [
        decision.opening_species_id,
        decision.closing_species_id,
        decision.strongest_species_id,
    ]:
        if selected_id not in allowed_set:
            raise ValueError(
                f"Invalid selected species ID: {selected_id}"
            )

    score_species_ids = {
        item.species_id
        for item in decision.final_scores
    }

    if score_species_ids != allowed_set:
        raise ValueError(
            "final_scores must contain "
            "all species IDs exactly once."
        )

    return (
        decision,
        decision_model_id,
    )


edit_decision, edit_decision_model_id = (
    generate_documentary_edit_decision()
)

print(
    "Edit decision model:",
    edit_decision_model_id,
)

print(
    edit_decision.model_dump_json(
        indent=2
    )
)

In [ ]:
all_media_rows = [
    *image_rows,
    *audio_rows,
    *video_rows,
]

assert len(image_rows) == 3
assert len(audio_rows) == 3
assert len(video_rows) == 3
assert len(all_media_rows) == 9

model_selection = {
    "portfolio_model": portfolio_model_id,
    "edit_decision_model": (
        edit_decision_model_id
    ),
    "image_models_used": sorted(
        {
            row["generation_model"]
            for row in image_rows
        }
    ),
    "image_resolutions_requested": sorted(
        {
            row["requested_resolution"]
            for row in image_rows
        }
    ),
    "audio_models_used": sorted(
        {
            row["generation_model"]
            for row in audio_rows
        }
    ),
    "video_models_used": sorted(
        {
            row["generation_model"]
            for row in video_rows
        }
    ),
    "video_resolutions_requested": sorted(
        {
            row["requested_resolution"]
            for row in video_rows
        }
    ),
    "review_models_used": sorted(
        {
            row["review_model"]
            for row in review_rows
        }
    ),
}

run_row = {
    "run_id": run_id,
    "documentary_title": (
        species_portfolio.documentary_title
    ),
    "structured_brief_json": (
        structured_brief_json
    ),
    "structured_portfolio_json": (
        structured_portfolio_json
    ),
    "edit_decision_json": (
        edit_decision.model_dump_json(
            indent=2
        )
    ),
    "model_selection_json": json.dumps(
        model_selection,
        indent=2,
        ensure_ascii=False,
    ),
    "created_at": (
        run_timestamp.isoformat()
    ),
}

review_rows.append(
    {
        "review_id": str(uuid4()),
        "run_id": run_id,
        "species_id": "documentary",
        "review_type": "documentary_edit_decision",
        "review_json": (
            edit_decision.model_dump_json(
                indent=2
            )
        ),
        "review_model": edit_decision_model_id,
        "created_at": (
            datetime.now(timezone.utc).isoformat()
        ),
    }
)

print(
    json.dumps(
        model_selection,
        indent=2,
        ensure_ascii=False,
    )
)

In [ ]:
run_load_result = batch_load_rows_to_bigquery(
    rows=[run_row],
    table_ref=run_table_ref,
    schema=run_schema,
    table_name=RUN_TABLE_ID,
    run_id=run_id,
)

species_load_result = batch_load_rows_to_bigquery(
    rows=species_rows,
    table_ref=species_table_ref,
    schema=species_schema,
    table_name=SPECIES_TABLE_ID,
    run_id=run_id,
)

media_load_result = batch_load_rows_to_bigquery(
    rows=all_media_rows,
    table_ref=media_table_ref,
    schema=media_schema,
    table_name=MEDIA_TABLE_ID,
    run_id=run_id,
)

review_load_result = batch_load_rows_to_bigquery(
    rows=review_rows,
    table_ref=review_table_ref,
    schema=review_schema,
    table_name=REVIEW_TABLE_ID,
    run_id=run_id,
)

print(run_load_result)
print(species_load_result)
print(media_load_result)
print(review_load_result)

In [ ]:
sql = f"""
SELECT
  media_type,
  requested_resolution,
  generation_model,
  COUNT(*) AS asset_count,
  SUM(size_bytes) AS total_bytes
FROM `{media_table_ref}`
WHERE run_id = @run_id
GROUP BY
  media_type,
  requested_resolution,
  generation_model
ORDER BY
  media_type,
  requested_resolution,
  generation_model
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter(
            "run_id",
            "STRING",
            run_id,
        )
    ]
)

media_verification_df = (
    bigquery_client.query(
        sql,
        job_config=job_config,
        location=dataset.location,
    )
    .to_dataframe()
)

display(
    media_verification_df
)

media_type_counts = (
    media_verification_df
    .groupby("media_type")["asset_count"]
    .sum()
    .to_dict()
)

assert media_type_counts["species_image"] == 3
assert media_type_counts["species_soundtrack"] == 3
assert media_type_counts["species_video"] == 3

print("Media validation passed.")
print("Images:", media_type_counts["species_image"])
print("Soundtracks:", media_type_counts["species_soundtrack"])
print("Videos:", media_type_counts["species_video"])

In [ ]:
score_by_species_id = {
    score.species_id: score.score
    for score in edit_decision.final_scores
}

documentary_segments = []

for species_id in edit_decision.recommended_sequence:
    species = find_species(
        species_id
    )

    image_row = next(
        row
        for row in image_rows
        if row["species_id"] == species_id
    )

    audio_row = next(
        row
        for row in audio_rows
        if row["species_id"] == species_id
    )

    video_row = next(
        row
        for row in video_rows
        if row["species_id"] == species_id
    )

    documentary_segments.append(
        {
            "species_id": species_id,
            "common_name": species.common_name,
            "scientific_name": species.scientific_name,
            "planet_name": species.planet_name,
            "habitat_type": species.habitat_type,
            "documentary_hook": species.documentary_hook,
            "image_gcs_uri": image_row["gcs_uri"],
            "image_resolution": (
                image_row["requested_resolution"]
            ),
            "image_dimensions": {
                "width": image_row["width"],
                "height": image_row["height"],
            },
            "audio_gcs_uri": audio_row["gcs_uri"],
            "video_gcs_uri": video_row["gcs_uri"],
            "video_resolution": (
                video_row["requested_resolution"]
            ),
            "video_duration_seconds": (
                video_row["duration_seconds"]
            ),
            "final_score": (
                score_by_species_id[species_id]
            ),
        }
    )

final_manifest = {
    "manifest_id": str(uuid4()),
    "run_id": run_id,
    "created_at": (
        datetime.now(timezone.utc).isoformat()
    ),
    "notebook": (
        "73_w_google_cloud_exoplanet_"
        "wildlife_documentary_4k_studio.ipynb"
    ),
    "documentary_brief": (
        documentary_brief.model_dump(
            mode="json"
        )
    ),
    "species_portfolio": (
        species_portfolio.model_dump(
            mode="json"
        )
    ),
    "edit_decision": (
        edit_decision.model_dump(
            mode="json"
        )
    ),
    "model_selection": model_selection,
    "asset_counts": {
        "images": len(image_rows),
        "soundtracks": len(audio_rows),
        "videos": len(video_rows),
        "total_assets": len(all_media_rows),
    },
    "documentary_segments": documentary_segments,
    "species_reviews": {
        species_id: review.model_dump(
            mode="json"
        )
        for species_id, review in species_reviews.items()
    },
    "local_files_saved": False,
}

manifest_json = json.dumps(
    final_manifest,
    indent=2,
    ensure_ascii=False,
    default=str,
)

manifest_blob_name = (
    f"{GCS_MANIFEST_PREFIX}/"
    f"{run_id}/"
    "exoplanet_documentary_manifest.json"
)

manifest_gcs_uri = upload_text_to_gcs(
    manifest_json,
    blob_name=manifest_blob_name,
    content_type="application/json",
)

print("Manifest saved to:")
print(manifest_gcs_uri)

In [ ]:
notebook_summary = {
    "project_id": PROJECT_ID,
    "global_location": GLOBAL_LOCATION,
    "video_location": VIDEO_LOCATION,
    "bucket_location": BUCKET_LOCATION,
    "bigquery_location": dataset.location,
    "run_id": run_id,
    "created_at": (
        datetime.now(timezone.utc).isoformat()
    ),
    "documentary_title": (
        species_portfolio.documentary_title
    ),
    "models": model_selection,
    "counts": {
        "species": len(species_rows),
        "images": len(image_rows),
        "soundtracks": len(audio_rows),
        "videos": len(video_rows),
        "reviews": len(review_rows),
    },
    "bigquery": {
        "run_table": run_table_ref,
        "species_table": species_table_ref,
        "media_table": media_table_ref,
        "review_table": review_table_ref,
    },
    "cloud_storage": {
        "manifest_gcs_uri": manifest_gcs_uri,
        "image_prefix": (
            f"gs://{BUCKET_NAME}/"
            f"{GCS_IMAGE_PREFIX}/{run_id}"
        ),
        "audio_prefix": (
            f"gs://{BUCKET_NAME}/"
            f"{GCS_AUDIO_PREFIX}/{run_id}"
        ),
        "video_prefix": (
            f"gs://{BUCKET_NAME}/"
            f"{GCS_VIDEO_PREFIX}/{run_id}"
        ),
        "local_files_saved": False,
    },
    "edit_decision": (
        edit_decision.model_dump(
            mode="json"
        )
    ),
    "segments": documentary_segments,
}

summary_json = json.dumps(
    notebook_summary,
    indent=2,
    ensure_ascii=False,
    default=str,
)

summary_blob_name = (
    f"{GCS_SUMMARY_PREFIX}/"
    f"{run_id}/"
    "summary.json"
)

summary_gcs_uri = upload_text_to_gcs(
    summary_json,
    blob_name=summary_blob_name,
    content_type="application/json",
)

print("Summary saved to:")
print(summary_gcs_uri)

In [ ]:
print(
    "EXOPLANET WILDLIFE DOCUMENTARY COMPLETED"
)

print("=" * 100)

print("Run ID:")
print(run_id)

print("\nDocumentary title:")
print(edit_decision.final_title)

print("\nExecutive summary:")
print(edit_decision.executive_summary)

print("\nRecommended sequence:")

for position, species_id in enumerate(
    edit_decision.recommended_sequence,
    start=1,
):
    species = find_species(
        species_id
    )

    image_row = next(
        row
        for row in image_rows
        if row["species_id"] == species_id
    )

    audio_row = next(
        row
        for row in audio_rows
        if row["species_id"] == species_id
    )

    video_row = next(
        row
        for row in video_rows
        if row["species_id"] == species_id
    )

    review = species_reviews[
        species_id
    ]

    print("\n" + "=" * 100)

    print(
        f"{position}. {species.common_name}"
    )

    print(
        "Scientific name:",
        species.scientific_name,
    )

    print(
        "Planet:",
        species.planet_name,
    )

    print(
        "Habitat:",
        species.habitat_type,
    )

    print(
        "Final score:",
        score_by_species_id[species_id],
        "/ 100",
    )

    print(
        "Documentary hook:",
        species.documentary_hook,
    )

    print("\nIMAGE")
    print(
        image_row["gcs_uri"]
    )

    print(
        "Resolution:",
        image_row["requested_resolution"],
    )

    print(
        "Dimensions:",
        image_row["width"],
        "x",
        image_row["height"],
    )

    display_gcs_image(
        image_row["gcs_uri"],
        width=720,
    )

    print("\nSOUNDTRACK")
    print(
        audio_row["gcs_uri"]
    )

    display_gcs_audio(
        audio_row["gcs_uri"]
    )

    print("\nVIDEO")
    print(
        video_row["gcs_uri"]
    )

    print(
        "Resolution:",
        video_row["requested_resolution"],
    )

    display_gcs_video(
        video_row["gcs_uri"],
        width=720,
    )

    print("\nMULTIMEDIA REVIEW")
    print(
        review.model_dump_json(
            indent=2
        )
    )

print("\n" + "=" * 100)

print("Strongest species:")
print(
    edit_decision.strongest_species_id
)

print(
    edit_decision.strongest_species_reason
)

print("\nGenerated assets:")
print("Images:", len(image_rows))
print("Soundtracks:", len(audio_rows))
print("Videos:", len(video_rows))

print("\nCloud outputs:")
print("Manifest:", manifest_gcs_uri)
print("Summary:", summary_gcs_uri)

print("\nBigQuery tables:")
print("-", run_table_ref)
print("-", species_table_ref)
print("-", media_table_ref)
print("-", review_table_ref)

print(
    "\nValidation passed: exactly "
    "3 images, 3 soundtracks and 3 videos."
)

print(
    "No local image, audio, video or data files were saved."
)